In [1]:
import os
import pandas as pd

base = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data"

filtered_file = os.path.join(
    base,
    "Ligands",
    "Properties",
    "Filtered_Ligands.xlsx"
)

df = pd.read_excel(filtered_file)

print(df.columns)
print(df.head())

Index(['Ligand_ID', 'SMILES', 'Molecular_Weight', 'LogP', 'H_Bond_Donors',
       'H_Bond_Acceptors', 'TPSA', 'Rotatable_Bonds'],
      dtype='object')
   Ligand_ID       SMILES  Molecular_Weight    LogP  H_Bond_Donors  \
0          1     CCC(C)CC            86.178  2.4425              0   
1          2  CC(C)CC(C)C           100.205  2.6885              0   
2          3       C=CCCC            70.135  1.9725              0   
3          4    C1=CCCCC1            82.146  2.1166              0   
4          5      C=CCC=C            68.119  1.7485              0   

   H_Bond_Acceptors  TPSA  Rotatable_Bonds  
0                 0   0.0                2  
1                 0   0.0                2  
2                 0   0.0                2  
3                 0   0.0                0  
4                 0   0.0                2  


In [2]:
import os

base = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data"

pdb_folder = os.path.join(base, "Ligands", "PDB")
pdbqt_folder = os.path.join(base, "Ligands", "PDBQT")

os.makedirs(pdb_folder, exist_ok=True)
os.makedirs(pdbqt_folder, exist_ok=True)

print("PDB Folder:", pdb_folder)
print("PDBQT Folder:", pdbqt_folder)

PDB Folder: C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDB
PDBQT Folder: C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT


In [3]:
print("Total ligands:", len(df))
print(df.head())

Total ligands: 237
   Ligand_ID       SMILES  Molecular_Weight    LogP  H_Bond_Donors  \
0          1     CCC(C)CC            86.178  2.4425              0   
1          2  CC(C)CC(C)C           100.205  2.6885              0   
2          3       C=CCCC            70.135  1.9725              0   
3          4    C1=CCCCC1            82.146  2.1166              0   
4          5      C=CCC=C            68.119  1.7485              0   

   H_Bond_Acceptors  TPSA  Rotatable_Bonds  
0                 0   0.0                2  
1                 0   0.0                2  
2                 0   0.0                2  
3                 0   0.0                0  
4                 0   0.0                2  


In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem
import os

# First ligand
row = df.iloc[0]

ligand_id = row["Ligand_ID"]
smiles = row["SMILES"]

print("Ligand ID:", ligand_id)
print("SMILES:", smiles)

# Create molecule
mol = Chem.MolFromSmiles(smiles)
print("Mol created:", mol is not None)

# Add hydrogens
mol = Chem.AddHs(mol)

# Generate 3D coordinates
status = AllChem.EmbedMolecule(mol, randomSeed=42)
print("Embed status:", status)

# Optimize geometry
opt_status = AllChem.MMFFOptimizeMolecule(mol)
print("Optimization status:", opt_status)

# Output file
output = os.path.join(pdb_folder, "test_ligand.pdb")

Chem.MolToPDBFile(mol, output)

print("Output file:", output)
print("File exists:", os.path.exists(output))

Ligand ID: 1
SMILES: CCC(C)CC
Mol created: True
Embed status: 0
Optimization status: 0
Output file: C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDB\test_ligand.pdb
File exists: True


In [5]:
from rdkit import Chem
from rdkit.Chem import AllChem
import os

success = 0
failed = 0

for index, row in df.iterrows():

    try:
        ligand_id = int(row["Ligand_ID"])
        smiles = row["SMILES"]

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            print(f"Invalid SMILES for Ligand {ligand_id}")
            failed += 1
            continue

        mol = Chem.AddHs(mol)

        status = AllChem.EmbedMolecule(mol, randomSeed=42)

        if status != 0:
            print(f"Embedding failed for Ligand {ligand_id}")
            failed += 1
            continue

        AllChem.MMFFOptimizeMolecule(mol)

        output = os.path.join(
            pdb_folder,
            f"Ligand_{ligand_id:04d}.pdb"
        )

        Chem.MolToPDBFile(mol, output)

        success += 1

    except Exception as e:
        print(f"Error for Ligand {ligand_id}: {e}")
        failed += 1

print("\nFinished!")
print("PDB files created:", success)
print("Failed:", failed)


Finished!
PDB files created: 237
Failed: 0


In [6]:
import os

files = sorted(os.listdir(pdb_folder))

print("Total PDB files:", len(files))
print(files[:10])

Total PDB files: 238
['Ligand_0001.pdb', 'Ligand_0002.pdb', 'Ligand_0003.pdb', 'Ligand_0004.pdb', 'Ligand_0005.pdb', 'Ligand_0006.pdb', 'Ligand_0007.pdb', 'Ligand_0008.pdb', 'Ligand_0009.pdb', 'Ligand_0010.pdb']


In [7]:
import os

file = os.path.join(pdb_folder, "Ligand_0104.pdb")

print(os.path.exists(file))
print(os.path.getsize(file))

True
5219


In [9]:
import os
import subprocess

obabel = r"C:\Software\anaconda3\envs\drug_discovery\Library\bin\obabel.exe"

pdb_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDB"

pdbqt_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT"

In [1]:
import meeko
import rdkit
import scipy

print("Meeko:", meeko.__version__)
print("RDKit:", rdkit.__version__)
print("SciPy:", scipy.__version__)

Meeko: 0.7.1
RDKit: 2026.03.4
SciPy: 1.15.3


In [2]:
import meeko

print(meeko.__version__)
print(dir(meeko))

0.7.1
['AtomTyper', 'Hydrate', 'MoleculePreparation', 'MoleculeSetup', 'Monomer', 'PDBQTMolecule', 'PDBQTReceptor', 'PDBQTWriterLegacy', 'Polymer', 'PolymerCreationError', 'RDKitMolCreate', 'RDKitMoleculeSetup', 'ResidueChemTemplates', 'ResiduePadder', 'ResidueTemplate', 'Restraint', 'UniqAtomParams', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_has_prody', 'add_rotamers_to_polymer_molsetups', 'analysis', 'atomtyper', 'bondtyper', 'chemtempgen', 'espalomatyper', 'export_flexres', 'export_pdb_updated_flexres', 'flexibility', 'geomutils', 'get_openff_epsilon_sigma', 'get_reactive_config', 'hydrate', 'load_openff', 'logging', 'macrocycle', 'molecule_pdbqt', 'molsetup', 'oids_block_from_setup', 'openff_xml_parser', 'parse_offxml', 'pdbutils', 'polymer', 'preparation', 'rdBase', 'rdkit_logger', 'rdkit_mol_create', 'rdkitutils', 'reactive', 'reactive_typer', 'receptor_pdbqt', 'utils', 'writer

In [3]:
import os

folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\Properties"

print(os.listdir(folder))

['Filtered_Ligands.xlsx']


In [4]:
import os

folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands"

for root, dirs, files in os.walk(folder):
    for file in files:
        print(os.path.join(root, file))

C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDB\Ligand_0001.pdb
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDB\Ligand_0002.pdb
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDB\Ligand_0003.pdb
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDB\Ligand_0004.pdb
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDB\Ligand_0005.pdb
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDB\Ligand_0006.pdb
C:\U

In [5]:
import os

project = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data"

for root, dirs, files in os.walk(project):
    for f in files:
        if f.endswith((".csv", ".xlsx", ".xls", ".sdf", ".smi", ".txt")):
            print(os.path.join(root, f))

C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\Properties\Filtered_Ligands.xlsx
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\Raw\drug_like_molecules.sdf
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Results\Binding_Site_Centers.xlsx
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Results\Clean_Protein_Analysis.xlsx
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Results\Protein_Properties.xlsx


In [6]:
from rdkit import Chem

sdf_file = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\Raw\drug_like_molecules.sdf"

supplier = Chem.SDMolSupplier(sdf_file)

molecules = [mol for mol in supplier if mol is not None]

print("Total molecules:", len(molecules))

Total molecules: 257


In [7]:
from rdkit import Chem

sdf_file = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\Raw\drug_like_molecules.sdf"

supplier = Chem.SDMolSupplier(sdf_file)
molecules = [m for m in supplier if m is not None]

mol = molecules[0]

print("Atoms:", mol.GetNumAtoms())
print("Conformers:", mol.GetNumConformers())

if mol.GetNumConformers() > 0:
    conf = mol.GetConformer()
    pos = conf.GetAtomPosition(0)
    print("First atom coordinates:")
    print(pos.x, pos.y, pos.z)

Atoms: 6
Conformers: 1
First atom coordinates:
0.2606 0.1503 0.0


In [8]:
from meeko import MoleculePreparation
from rdkit import Chem

prep = MoleculePreparation()

print("MoleculePreparation created successfully!")

print("\nMethods available:")
print([m for m in dir(prep) if not m.startswith("_")])

MoleculePreparation created successfully!

Methods available:
['add_atom_types', 'add_index_map', 'atom_params', 'calc_flex', 'charge_atom_prop', 'charge_model', 'check_external_ring_break', 'deprecated_setup_access', 'dihedral_model', 'dihedral_params', 'double_bond_penalty', 'flexible_amides', 'from_config', 'from_json_file', 'get_atom_params', 'get_defaults_dict', 'hydrate', 'input_atom_params', 'keep_chorded_rings', 'keep_equivalent_rings', 'load_atom_params', 'load_offatom_params', 'macrocycle_allow_A', 'max_ring_size', 'merge_these_atom_types', 'min_ring_size', 'name', 'offatom_params', 'packaged_params', 'path', 'prepare', 'reactive_smarts', 'reactive_smarts_idx', 'remove_smiles', 'rigid_macrocycles', 'rigidify_bonds_indices', 'rigidify_bonds_smarts', 'setup', 'untyped_macrocycles', 'write_pdbqt_file', 'write_pdbqt_string']


In [9]:
import os
from rdkit import Chem
from meeko import MoleculePreparation

In [10]:
# SDF file
sdf_file = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\Raw\drug_like_molecules.sdf"

# Output folder
output_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT"

os.makedirs(output_folder, exist_ok=True)

print("Output Folder:")
print(output_folder)

Output Folder:
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT


In [11]:
supplier = Chem.SDMolSupplier(sdf_file)

molecules = [mol for mol in supplier if mol is not None]

print("Total molecules:", len(molecules))

Total molecules: 257


In [14]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation

mol = molecules[0]

# Add explicit hydrogens
mol = Chem.AddHs(mol)

# Optimize geometry (optional but recommended)
AllChem.MMFFOptimizeMolecule(mol)

prep = MoleculePreparation()

prep.prepare(mol)

outfile = os.path.join(output_folder, "Ligand_0001.pdbqt")

prep.write_pdbqt_file(outfile)

print("Saved:", outfile)

Saved: C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT\Ligand_0001.pdbqt


C:\Software\anaconda3\envs\drug_discovery\lib\site-packages\meeko\molsetup.py:1584: RuntimeWarning: RDKit molecule not labeled as 3D. This warning won't show again.
  warnings.warn(
C:\Software\anaconda3\envs\drug_discovery\lib\site-packages\meeko\preparation.py:714: DeprecationWarning: MoleculePreparation.write_pdbqt_file() is deprecated since Meeko v0.5
  warnings.warn(
C:\Software\anaconda3\envs\drug_discovery\lib\site-packages\meeko\preparation.py:693: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
C:\Software\anaconda3\envs\drug_discovery\lib\site-packages\meeko\preparation.py:467: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, Deprecation

In [15]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation

# -------------------------
# Paths
# -------------------------
sdf_file = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\Raw\drug_like_molecules.sdf"

output_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT"

os.makedirs(output_folder, exist_ok=True)

# -------------------------
# Read SDF
# -------------------------
supplier = Chem.SDMolSupplier(sdf_file)

success = 0
failed = 0

for i, mol in enumerate(supplier):

    if mol is None:
        failed += 1
        print(f"❌ Molecule {i+1} could not be read.")
        continue

    outfile = os.path.join(
        output_folder,
        f"Ligand_{i+1:04d}.pdbqt"
    )

    # Skip if already exists
    if os.path.exists(outfile):
        print(f"Skipped: Ligand_{i+1:04d}")
        continue

    try:

        # Add hydrogens
        mol = Chem.AddHs(mol)

        # Geometry optimization
        AllChem.MMFFOptimizeMolecule(mol)

        # Prepare ligand
        prep = MoleculePreparation()
        prep.prepare(mol)

        # Save
        prep.write_pdbqt_file(outfile)

        success += 1

        print(f"✅ {i+1}/257")

    except Exception as e:

        failed += 1

        print(f"❌ Ligand {i+1}")
        print(e)

print("\n==========================")
print("Finished!")
print("==========================")
print("Successful :", success)
print("Failed     :", failed)

Skipped: Ligand_0001
Skipped: Ligand_0002
Skipped: Ligand_0003
Skipped: Ligand_0004
Skipped: Ligand_0005
Skipped: Ligand_0006
Skipped: Ligand_0007
Skipped: Ligand_0008
Skipped: Ligand_0009
Skipped: Ligand_0010
Skipped: Ligand_0011
Skipped: Ligand_0012
Skipped: Ligand_0013
Skipped: Ligand_0014
Skipped: Ligand_0015
Skipped: Ligand_0016
Skipped: Ligand_0017
✅ 18/257
✅ 19/257
Skipped: Ligand_0020
Skipped: Ligand_0021
✅ 22/257
Skipped: Ligand_0023
Skipped: Ligand_0024
Skipped: Ligand_0025
Skipped: Ligand_0026
Skipped: Ligand_0027
Skipped: Ligand_0028
Skipped: Ligand_0029
Skipped: Ligand_0030
Skipped: Ligand_0031
Skipped: Ligand_0032
Skipped: Ligand_0033
Skipped: Ligand_0034
Skipped: Ligand_0035
✅ 36/257
Skipped: Ligand_0037
Skipped: Ligand_0038
Skipped: Ligand_0039
Skipped: Ligand_0040
Skipped: Ligand_0041
Skipped: Ligand_0042
Skipped: Ligand_0043
Skipped: Ligand_0044
Skipped: Ligand_0045
✅ 46/257
✅ 47/257
✅ 48/257
✅ 49/257
✅ 50/257


[RDKit] ERROR:[22:55:37] 
[RDKit] ERROR:
[RDKit] ERROR:****
[RDKit] ERROR:Invariant Violation
[RDKit] ERROR:bad direction in linearSearch
[RDKit] ERROR:Violation occurred on line 234 in file D:\bld\rdkit-meta_1783709451439\work\Code\Numerics/Optimizer/BFGSOpt.h
[RDKit] ERROR:Failed Expression: status >= 0
[RDKit] ERROR:****
[RDKit] ERROR:
[RDKit] ERROR:[22:55:37] 
[RDKit] ERROR:
[RDKit] ERROR:****
[RDKit] ERROR:Invariant Violation
[RDKit] ERROR:bad direction in linearSearch
[RDKit] ERROR:Violation occurred on line 234 in file D:\bld\rdkit-meta_1783709451439\work\Code\Numerics/Optimizer/BFGSOpt.h
[RDKit] ERROR:Failed Expression: status >= 0
[RDKit] ERROR:****
[RDKit] ERROR:


✅ 51/257
✅ 52/257
✅ 53/257
✅ 54/257
✅ 55/257
Skipped: Ligand_0056
Skipped: Ligand_0057
Skipped: Ligand_0058
Skipped: Ligand_0059
Skipped: Ligand_0060
Skipped: Ligand_0061
Skipped: Ligand_0062
Skipped: Ligand_0063
Skipped: Ligand_0064
Skipped: Ligand_0065
Skipped: Ligand_0066
Skipped: Ligand_0067
Skipped: Ligand_0068
Skipped: Ligand_0069
Skipped: Ligand_0070
Skipped: Ligand_0071
Skipped: Ligand_0072
Skipped: Ligand_0073
Skipped: Ligand_0074
Skipped: Ligand_0075
Skipped: Ligand_0076
Skipped: Ligand_0077
Skipped: Ligand_0078
Skipped: Ligand_0079
Skipped: Ligand_0080
Skipped: Ligand_0081
Skipped: Ligand_0082
Skipped: Ligand_0083
Skipped: Ligand_0084
Skipped: Ligand_0085
Skipped: Ligand_0086
Skipped: Ligand_0087
Skipped: Ligand_0088
Skipped: Ligand_0089
Skipped: Ligand_0090
Skipped: Ligand_0091
Skipped: Ligand_0092
Skipped: Ligand_0093
Skipped: Ligand_0094
Skipped: Ligand_0095
Skipped: Ligand_0096
Skipped: Ligand_0097
Skipped: Ligand_0098
Skipped: Ligand_0099
Skipped: Ligand_0100
Skipped: L

[RDKit] ERROR:[22:55:37] 
[RDKit] ERROR:
[RDKit] ERROR:****
[RDKit] ERROR:Invariant Violation
[RDKit] ERROR:bad direction in linearSearch
[RDKit] ERROR:Violation occurred on line 234 in file D:\bld\rdkit-meta_1783709451439\work\Code\Numerics/Optimizer/BFGSOpt.h
[RDKit] ERROR:Failed Expression: status >= 0
[RDKit] ERROR:****
[RDKit] ERROR:
[RDKit] ERROR:[22:55:37] 
[RDKit] ERROR:
[RDKit] ERROR:****
[RDKit] ERROR:Invariant Violation
[RDKit] ERROR:bad direction in linearSearch
[RDKit] ERROR:Violation occurred on line 234 in file D:\bld\rdkit-meta_1783709451439\work\Code\Numerics/Optimizer/BFGSOpt.h
[RDKit] ERROR:Failed Expression: status >= 0
[RDKit] ERROR:****
[RDKit] ERROR:
[RDKit] ERROR:[22:55:37] 


✅ 113/257
✅ 114/257
❌ Ligand 115
Invariant Violation
	bad direction in linearSearch
	Violation occurred on line 234 in file Code\Numerics/Optimizer/BFGSOpt.h
	Failed Expression: status >= 0
	RDKIT: 2026.03.4
	BOOST: 

✅ 116/257
✅ 117/257
✅ 118/257
❌ Ligand 119
Invariant Violation
	bad direction in linearSearch
	Violation occurred on line 234 in file Code\Numerics/Optimizer/BFGSOpt.h
	Failed Expression: status >= 0
	RDKIT: 2026.03.4
	BOOST: 

✅ 120/257
✅ 121/257
✅ 122/257
✅ 123/257
✅ 124/257


...

✅ 244/257
✅ 245/257
✅ 246/257
❌ Ligand 247
Invariant Violation
	bad direction in linearSearch
	Violation occurred on line 234 in file Code\Numerics/Optimizer/BFGSOpt.h
	Failed Expression: status >= 0
	RDKIT: 2026.03.4
	BOOST: 

✅ 248/257


[RDKit] ERROR:[22:55:41] 
[RDKit] ERROR:
[RDKit] ERROR:****
[RDKit] ERROR:Invariant Violation
[RDKit] ERROR:bad direction in linearSearch
[RDKit] ERROR:Violation occurred on line 234 in file D:\bld\rdkit-meta_1783709451439\work\Code\Numerics/Optimizer/BFGSOpt.h
[RDKit] ERROR:Failed Expression: status >= 0
[RDKit] ERROR:****
[RDKit] ERROR:


✅ 249/257
✅ 250/257
❌ Ligand 251
Invariant Violation
	bad direction in linearSearch
	Violation occurred on line 234 in file Code\Numerics/Optimizer/BFGSOpt.h
	Failed Expression: status >= 0
	RDKIT: 2026.03.4
	BOOST: 

✅ 252/257


[RDKit] ERROR:[22:55:41] 
[RDKit] ERROR:
[RDKit] ERROR:****
[RDKit] ERROR:Invariant Violation
[RDKit] ERROR:bad direction in linearSearch
[RDKit] ERROR:Violation occurred on line 234 in file D:\bld\rdkit-meta_1783709451439\work\Code\Numerics/Optimizer/BFGSOpt.h
[RDKit] ERROR:Failed Expression: status >= 0
[RDKit] ERROR:****
[RDKit] ERROR:


✅ 253/257
✅ 254/257
❌ Ligand 255
Invariant Violation
	bad direction in linearSearch
	Violation occurred on line 234 in file Code\Numerics/Optimizer/BFGSOpt.h
	Failed Expression: status >= 0
	RDKIT: 2026.03.4
	BOOST: 

✅ 256/257
✅ 257/257

Finished!
Successful : 130
Failed     : 37


In [16]:
import os

folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT"

files = [f for f in os.listdir(folder) if f.endswith(".pdbqt")]

print("Total PDBQT files:", len(files))

Total PDBQT files: 221


In [17]:
import os

output_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT"

missing = []

for i in range(1, 258):
    filename = f"Ligand_{i:04d}.pdbqt"
    if not os.path.exists(os.path.join(output_folder, filename)):
        missing.append(i)

print("Missing ligands:", missing)
print("\nTotal missing:", len(missing))

Missing ligands: [111, 112, 115, 119, 125, 127, 128, 131, 134, 136, 137, 138, 147, 148, 150, 151, 156, 157, 163, 164, 176, 177, 181, 185, 189, 190, 196, 208, 214, 216, 217, 224, 230, 241, 247, 251, 255]

Total missing: 37


In [18]:
import os
from rdkit import Chem
from meeko import MoleculePreparation

# Paths
sdf_file = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\Raw\drug_like_molecules.sdf"

output_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT"

missing = [111,112,115,119,125,127,128,131,134,136,137,138,
           147,148,150,151,156,157,163,164,176,177,181,
           185,189,190,196,208,214,216,217,224,230,241,
           247,251,255]

supplier = Chem.SDMolSupplier(sdf_file)

for i, mol in enumerate(supplier, start=1):

    if i not in missing:
        continue

    if mol is None:
        print(f"Cannot read Ligand {i}")
        continue

    try:
        mol = Chem.AddHs(mol)

        prep = MoleculePreparation()
        prep.prepare(mol)

        outfile = os.path.join(output_folder, f"Ligand_{i:04d}.pdbqt")
        prep.write_pdbqt_file(outfile)

        print(f"Recovered Ligand {i}")

    except Exception as e:
        print(f"Failed Ligand {i}")
        print(e)

Recovered Ligand 111
Recovered Ligand 112
Recovered Ligand 115
Recovered Ligand 119
Recovered Ligand 125
Recovered Ligand 127
Recovered Ligand 128
Recovered Ligand 131
Recovered Ligand 134
Recovered Ligand 136
Recovered Ligand 137
Recovered Ligand 138
Recovered Ligand 147
Recovered Ligand 148
Recovered Ligand 150
Recovered Ligand 151
Recovered Ligand 156
Recovered Ligand 157
Recovered Ligand 163
Recovered Ligand 164
Recovered Ligand 176
Recovered Ligand 177
Recovered Ligand 181
Recovered Ligand 185
Recovered Ligand 189
Recovered Ligand 190
Recovered Ligand 196
Recovered Ligand 208
Recovered Ligand 214
Recovered Ligand 216
Recovered Ligand 217
Recovered Ligand 224
Recovered Ligand 230
Recovered Ligand 241
Recovered Ligand 247
Recovered Ligand 251
Recovered Ligand 255


In [19]:
import os

folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT"

files = [f for f in os.listdir(folder) if f.endswith(".pdbqt")]

print("Total PDBQT files:", len(files))

Total PDBQT files: 258


In [20]:
import os

folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT"

files = sorted([f for f in os.listdir(folder) if f.endswith(".pdbqt")])

print("Total:", len(files))

for f in files:
    print(f)

Total: 258
Ligand_0001.pdbqt
Ligand_0002.pdbqt
Ligand_0003.pdbqt
Ligand_0004.pdbqt
Ligand_0005.pdbqt
Ligand_0006.pdbqt
Ligand_0007.pdbqt
Ligand_0008.pdbqt
Ligand_0009.pdbqt
Ligand_0010.pdbqt
Ligand_0011.pdbqt
Ligand_0012.pdbqt
Ligand_0013.pdbqt
Ligand_0014.pdbqt
Ligand_0015.pdbqt
Ligand_0016.pdbqt
Ligand_0017.pdbqt
Ligand_0018.pdbqt
Ligand_0019.pdbqt
Ligand_0020.pdbqt
Ligand_0021.pdbqt
Ligand_0022.pdbqt
Ligand_0023.pdbqt
Ligand_0024.pdbqt
Ligand_0025.pdbqt
Ligand_0026.pdbqt
Ligand_0027.pdbqt
Ligand_0028.pdbqt
Ligand_0029.pdbqt
Ligand_0030.pdbqt
Ligand_0031.pdbqt
Ligand_0032.pdbqt
Ligand_0033.pdbqt
Ligand_0034.pdbqt
Ligand_0035.pdbqt
Ligand_0036.pdbqt
Ligand_0037.pdbqt
Ligand_0038.pdbqt
Ligand_0039.pdbqt
Ligand_0040.pdbqt
Ligand_0041.pdbqt
Ligand_0042.pdbqt
Ligand_0043.pdbqt
Ligand_0044.pdbqt
Ligand_0045.pdbqt
Ligand_0046.pdbqt
Ligand_0047.pdbqt
Ligand_0048.pdbqt
Ligand_0049.pdbqt
Ligand_0050.pdbqt
Ligand_0051.pdbqt
Ligand_0052.pdbqt
Ligand_0053.pdbqt
Ligand_0054.pdbqt
Ligand_0055.pdbqt

In [1]:
import os

base = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)"

docking = os.path.join(base, "Docking")

proteins = ["SDHA", "SDHB", "SDHC", "SDHD"]

for p in proteins:
    os.makedirs(os.path.join(docking, p), exist_ok=True)

print("Docking folders created successfully!")

Docking folders created successfully!


In [3]:
import os

protein_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Proteins"

files = sorted([f for f in os.listdir(protein_folder) if f.endswith(".pdbqt")])

print("Total Protein PDBQT files:", len(files))
print()

for file in files:
    print(file)

Total Protein PDBQT files: 1

6C12_clean.pdbqt


In [5]:
import os

pdb_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Proteins"

files = sorted([f for f in os.listdir(pdb_folder) if f.endswith(".pdb")])

print("Total PDB files:", len(files))
print()

for f in files:
    print(f)

Total PDB files: 4

1NEN.pdb
1Y8P.pdb
2H89.pdb
6C12.pdb


In [1]:
try:
    from meeko import MoleculePreparation
    print("✅ Meeko imported successfully!")
except Exception as e:
    print("❌ Error:", e)

✅ Meeko imported successfully!


In [2]:
import os

protein_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\PDBQT_Proteins"

files = sorted([f for f in os.listdir(protein_folder) if f.endswith(".pdbqt")])

print("Total PDBQT files:", len(files))
for f in files:
    print(f)

Total PDBQT files: 4
1NEN.pdbqt
1Y8P.pdbqt
2H89.pdbqt
6C12.pdbqt


In [4]:
import os

protein_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\PDBQT_Proteins"

for f in sorted(os.listdir(protein_folder)):
    print(os.path.join(protein_folder, f))

C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\PDBQT_Proteins\1NEN.pdbqt
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\PDBQT_Proteins\1Y8P.pdbqt
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\PDBQT_Proteins\2H89.pdbqt
C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\PDBQT_Proteins\6C12.pdbqt


In [5]:
import os

ligand_folder = r"C:\Users\sofiya chavarekar\OneDrive\Assignment- Module 1\PROJECT(AI-based drug target identification for SDH-related rare disorders)\Data\Ligands\PDBQT"

ligands = sorted([f for f in os.listdir(ligand_folder) if f.endswith(".pdbqt")])

print("Total ligands:", len(ligands))

print("\nFirst 5 ligands:")
for ligand in ligands[:5]:
    print(ligand)

Total ligands: 257

First 5 ligands:
Ligand_0001.pdbqt
Ligand_0002.pdbqt
Ligand_0003.pdbqt
Ligand_0004.pdbqt
Ligand_0005.pdbqt
